# AGEMR B3 v4：BMI × BP 年龄分布交叉分析

BMI source 和 BP target 各抽取五种年龄构成。每种 BMI 构成有三次配对重复，五个 BP 样本各抽取一次，SBP 和 DBP 使用相同的 BP 人员。分别汇总 BMI→SBP 和 BMI→DBP 的 25 种组合。

两侧用同一个 BP reference 年龄分布设置 5 岁年龄箱和指数倾斜参数。两侧分别在各自的人员池内设等样本量。若年龄箱容量有限，缩小该侧所有方案的共同样本量。BMI 与 BP 人员互不重叠，年龄方案之间可以共享同一侧的人。

每次 BMI source GWAS 从已有的候选面板按 P < 1e-3 选工具。对同一个 BMI source，五种 BP target 都使用这套候选工具，BP 关联无法估计的 SNP 按目标组记录。两侧的回归使用已保存的 nuisance covariates。对照方法为 IVW、dIVW、weighted median 和 MR-Egger。

在 Workbench 对已有 B1 结果运行本 notebook，结果写入 B3_MR_age_crossed 下的新目录。

In [ ]:
from pathlib import Path
import json, hashlib, uuid, subprocess, shutil, warnings, gc, tempfile, os
from datetime import datetime, timezone
import numpy as np
import pandas as pd
from scipy import stats, linalg, sparse
import matplotlib.pyplot as plt
from IPython.display import display

# Only this configuration cell normally needs editing.
LINEAR_RUN = None
B1_ROOT = Path.home()/"workspace/agemr-data/final_cohort_v1/B1_age_varying"
PLINK_EXECUTABLE = None  # e.g. str(Path.home()/"plink2_env/bin/plink2")
INSTALL_PLINK_IF_MISSING = True
PLINK_ENV = Path(tempfile.gettempdir())/("agemr-tools-"+str(os.getuid()))/"plink2_2.0.0a.7.8"
UNRELATED_KEEP = None  # Optional existing IID or FID=0 keep; no new kinship inference.
SEED = 20261028
N_TARGET_REQUESTED = 30000
N_SOURCE_REQUESTED = 30000
N_REPEATS = 3
N_BOOT = 2000
IV_P_THRESHOLD = 1e-3  # Sole relevance filter: current source ordinary GWAS P, strict <.
MAX_UNION_IV = 128  # Covariance memory guard only; do not truncate selected SNPs.
MIN_ROLE_N = 100
THREADS = 2
MEMORY_MB = 1024
AGE_BIN_YEARS = 5
SCENARIOS = [("matched", 0.), ("younger_moderate", -.5),
             ("younger_strong", -1.), ("older_moderate", .5), ("older_strong", 1.)]
OUTCOMES = ("SBP", "DBP")
METHODS = ("IVW", "dIVW", "weighted_median", "MR_Egger")
KEY = ["#CHROM", "POS", "ID", "REF", "ALT"]
CONTRAST_FAMILY = len(OUTCOMES)*len(METHODS)*2*len(SCENARIOS)*(len(SCENARIOS)-1)

def require(ok, message):
    if not ok: raise RuntimeError(message)

def sha(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for b in iter(lambda:f.read(1024*1024), b""): h.update(b)
    return h.hexdigest()

def write_json(path, value):
    Path(path).write_text(json.dumps(value, indent=2, default=str))

def iid_table(path):
    x = pd.read_csv(path, sep=r"\s+", dtype=str, keep_default_na=False)
    x = x.rename(columns={"#IID":"IID", "#FID":"FID"})
    require("IID" in x and x.IID.is_unique, f"Missing/duplicate IID: {path}")
    if "FID" in x:
        require(x.FID.eq("0").all(), "This workflow requires IID-only or FID=0 inputs")
        x = x.drop(columns="FID")
    return x.set_index("IID")

def stable_rng(label):
    value = int.from_bytes(hashlib.sha256(f"{SEED}|{label}".encode()).digest()[:8], "big")
    return np.random.default_rng(value)

def role(iid):
    u = int.from_bytes(hashlib.sha256(f"B3|{SEED}|{iid}".encode()).digest()[:8], "big")/2**64
    return "target" if .35 <= u < .55 else "source"

def run_cmd(command, prefix):
    prefix = Path(prefix); prefix.parent.mkdir(parents=True, exist_ok=True)
    command = list(map(str, command))
    write_json(str(prefix)+".command.json", command)
    with Path(str(prefix)+".stdout.txt").open("w") as out, Path(str(prefix)+".stderr.txt").open("w") as err:
        res = subprocess.run(command, stdout=out, stderr=err)
    if res.returncode:
        detail = Path(str(prefix)+".stderr.txt").read_text(errors="replace")[-2500:]
        raise RuntimeError(f"PLINK failed ({res.returncode}): {prefix}\n{detail}")

def nuisance_basis(C):
    X = np.column_stack([np.ones(len(C)), np.asarray(C, float)])
    Q, R, piv = linalg.qr(X, mode="economic", pivoting=True, check_finite=False)
    diag = np.abs(np.diag(R))
    rank = int((diag > max(X.shape)*np.finfo(float).eps*max(diag.max(), 1)).sum())
    return Q[:, :rank], rank, piv[:rank]

require(N_BOOT >= 200 and N_REPEATS >= 1, "Use at least 200 draws and one repeat")
if LINEAR_RUN is None:
    choices = []
    for marker in (B1_ROOT/"B1_linear_age").glob("*/RUN_COMPLETE.json"):
        m = json.loads(marker.read_text())
        if m.get("status") == "complete" and all((marker.parent/"inputs"/f"{t}.meta.json").is_file() for t in ("BMI",)+OUTCOMES):
            choices.append((m.get("finished_utc", ""), marker.parent))
    require(bool(choices), "No completed B1 linear run; set LINEAR_RUN to the existing run")
    LINEAR_RUN = max(choices, key=lambda z:z[0])[1]
else: LINEAR_RUN = Path(LINEAR_RUN).expanduser()
completion = json.loads((LINEAR_RUN/"RUN_COMPLETE.json").read_text())
require(completion.get("status") == "complete", "Selected run is incomplete")
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")+"_"+uuid.uuid4().hex[:6]
OUT = LINEAR_RUN/"B3_MR_age_crossed"/stamp
for name in ("qc", "design", "source_GWAS", "genotypes", "associations", "MR", "figures"):
    (OUT/name).mkdir(parents=True, exist_ok=True)
print("B3 v4 crossed age distributions - existing results:", LINEAR_RUN, flush=True)
print("New B3 outputs and software diagnostics:", OUT, flush=True)

# Embedded in each delivered notebook; no external Python file is needed.
def resolve_existing_or_install_plink(preferred, linear_run, log_dir, install_if_missing, env_prefix):
    import os, re, sys, tempfile
    log_dir = Path(log_dir); log_dir.mkdir(parents=True, exist_ok=True)
    env_prefix = Path(env_prefix).expanduser()
    audit, candidates, seen = [], [], set()

    def add(value, origin):
        if value is not None and str(value).strip():
            value = os.path.expandvars(str(value))
            found = shutil.which(value) if not Path(value).is_absolute() else None
            path = str(Path(found or value).expanduser().absolute())
            if path not in seen:
                seen.add(path); candidates.append((path, origin))

    def save():
        (log_dir/"plink_discovery.json").write_text(json.dumps(audit, indent=2))

    add(preferred, "explicit configuration")
    # Read program paths only; never replay a historical command or use shell=True.
    run = Path(linear_run)
    old_audit = run/"qc/plink_discovery.json"
    if old_audit.is_file():
        try:
            old = json.loads(old_audit.read_text())
            if isinstance(old, list):
                for r in old:
                    if isinstance(r, dict) and r.get("status") == "usable":
                        add(r.get("path"), "B1 successful discovery record")
        except (OSError, ValueError): pass
    historical = list(run.glob("*_results/*.command.json"))
    historical += list((run/"main_effect_stratified_proportions/pooled_main").glob("*/*.successful_signature.json"))
    for path in sorted(historical)[:200]:
        try:
            entry = json.loads(path.read_text())
            command = entry.get("command") if isinstance(entry, dict) else entry
            if isinstance(command,list) and command and isinstance(command[0],str):
                if Path(command[0]).name.lower() in ("plink2", "plink2.exe"):
                    add(command[0], "previous successful/run command: "+str(path))
        except (OSError, ValueError): continue
    for p in [shutil.which("plink2"), env_prefix/"bin/plink2",
              Path.home()/"plink2_env/bin/plink2", Path(sys.prefix)/"bin/plink2",
              Path.home()/".local/bin/plink2", "/home/dataproc/plink2_env/bin/plink2",
              "/home/jupyter/plink2_env/bin/plink2", "/opt/conda/default/bin/plink2",
              "/opt/conda/bin/plink2", "/usr/local/bin/plink2", "/usr/bin/plink2"]:
        add(p, "known software location")
    if env_prefix.parent.is_dir():
        for p in sorted(env_prefix.parent.glob("plink2_*/bin/plink2"))[:50]:
            add(p, "cached driver-local software")

    def probe(path, origin):
        r = {"path":str(path), "origin":origin}
        try:
            if not Path(path).is_file(): r["status"] = "missing"
            else:
                r["executable_mode"] = os.access(path,os.X_OK)
                v = subprocess.run([str(path),"--version"],capture_output=True,text=True,timeout=30)
                r["returncode"] = v.returncode
                r["version_or_error"] = (v.stdout+v.stderr).strip()[-2500:]
                r["status"] = "usable" if v.returncode == 0 and re.search(r"PLINK\s+v?2\.",r["version_or_error"],re.I) else "startup_failed"
        except (OSError, subprocess.TimeoutExpired) as e:
            r.update(status="execution_failed", error=str(e), errno=getattr(e,"errno",None))
        audit.append(r); save()
        if r["status"] == "usable": return str(path)
        return None

    def finish(path):
        print("Using PLINK2:",path,flush=True)
        print(audit[-1].get("version_or_error",""),flush=True)
        (log_dir/"plink_selected.json").write_text(json.dumps(audit[-1],indent=2))
        return path

    for path,origin in candidates:
        found = probe(path,origin)
        if found: return finish(found)
    important = [r for r in audit if r["status"] != "missing"]
    print("PLINK2 search diagnostics:",flush=True)
    print(pd.DataFrame(important or audit).to_string(index=False),flush=True)
    if not install_if_missing:
        raise RuntimeError(f"No runnable PLINK2; diagnostics: {log_dir/'plink_discovery.json'}. Set PLINK_EXECUTABLE or INSTALL_PLINK_IF_MISSING=True.")

    manager = None
    for p in [shutil.which("mamba"), shutil.which("conda"),
              "/opt/conda/default/bin/mamba", "/opt/conda/default/bin/conda", "/opt/conda/bin/conda"]:
        if p and Path(p).is_file() and os.access(p,os.X_OK): manager = str(p); break
    if manager is None:
        raise RuntimeError(f"No runnable PLINK2 and no mamba/conda available. Inspect {log_dir/'plink_discovery.json'}; set PLINK_EXECUTABLE to the working binary.")
    # Do not overwrite a broken prefix or mutate the Spark/Python environment.
    destination = env_prefix
    if destination.exists(): destination = destination.with_name(destination.name+"_"+uuid.uuid4().hex[:8])
    destination.parent.mkdir(parents=True,exist_ok=True)
    command = [manager,"create","-y","-p",str(destination),"--override-channels",
               "-c","conda-forge","-c","bioconda","plink2=2.0.0a.7.8"]
    (log_dir/"plink_install.command.json").write_text(json.dumps(command,indent=2))
    print("Installing public PLINK2 software only into driver-local:",destination,flush=True)
    print("Genotype/phenotype files remain in the workspace; active Python/Spark environment is unchanged.",flush=True)
    with (log_dir/"plink_install.log").open("w") as log:
        proc = subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        try:
            for line in proc.stdout:
                print(line,end="",flush=True); log.write(line); log.flush()
            returncode = proc.wait()
        except BaseException:
            proc.terminate()
            try: proc.wait(timeout=10)
            except subprocess.TimeoutExpired: proc.kill(); proc.wait()
            raise
    if returncode:
        raise RuntimeError(f"Public software installation failed ({returncode}). See {log_dir/'plink_install.log'}; no genotype analysis has started.")
    found = probe(destination/"bin/plink2","new public software installation")
    if not found:
        raise RuntimeError("PLINK2 was installed but cannot start: "+json.dumps(audit[-1])+f". See {log_dir/'plink_discovery.json'}")
    return finish(found)

PLINK = resolve_existing_or_install_plink(PLINK_EXECUTABLE,LINEAR_RUN,OUT/"qc",INSTALL_PLINK_IF_MISSING,PLINK_ENV)
PLINK_VERSION = json.loads((OUT/"qc/plink_selected.json").read_text())["version_or_error"]

records = json.loads((LINEAR_RUN/"qc/input_files.json").read_text())
pfiles = {int(Path(r["path"]).stem.replace("BMI_chr", "")):Path(r["path"]).with_suffix("")
          for r in records if Path(r["path"]).suffix == ".pgen"}
for record in records:
    p = Path(record["path"])
    if p.suffix in (".pgen", ".pvar", ".psam"):
        st = p.stat()
        require(st.st_size == record["bytes"] and st.st_mtime_ns == record["mtime_ns"], f"Genotype input changed: {p}")
panel = pd.read_csv(LINEAR_RUN/"qc/variant_panel.tsv", sep="\t", dtype={"#CHROM":str})
panel["#CHROM"] = panel["#CHROM"].str.replace(r"^chr", "", regex=True)
require(panel.ID.is_unique and not panel.duplicated(KEY).any(), "Ambiguous panel identifiers")
require(panel["#CHROM"].astype(int).between(1,22).all() and not panel.ALT.str.contains(",").any(), "Autosomal biallelic panel required")

pheno, covar, age, input_audit = {}, {}, {}, []
unrelated = None if UNRELATED_KEEP is None else set(iid_table(Path(UNRELATED_KEEP)).index)
for trait in ("BMI",)+OUTCOMES:
    pp = LINEAR_RUN/"inputs"/f"{trait}.pheno.tsv"
    cp = LINEAR_RUN/"inputs"/f"{trait}.covar.tsv"
    mp = LINEAR_RUN/"inputs"/f"{trait}.meta.json"
    meta = json.loads(mp.read_text())
    require(sha(pp) == meta["pheno_sha256"] and sha(cp) == meta["covar_sha256"], f"{trait}: saved input changed")
    ph = iid_table(pp)[[trait]].apply(pd.to_numeric, errors="raise").astype(float)
    cv = iid_table(cp).loc[ph.index].apply(pd.to_numeric, errors="raise").astype(float)
    if unrelated is not None: ph = ph.loc[ph.index.isin(unrelated)]; cv = cv.loc[ph.index]
    require(np.isfinite(ph.to_numpy()).all() and np.isfinite(cv.to_numpy()).all(), f"Missing {trait} input")
    require("AGE10" in cv and meta["age_unit_years"] == 10, "Need saved AGE10 and age center metadata")
    pheno[trait], covar[trait] = ph[trait], cv
    age[trait] = cv.AGE10*float(meta["age_unit_years"])+float(meta["age_center_years"])
    input_audit.append({"trait":trait, "N":len(ph), "pheno_sha256":sha(pp), "covar_sha256":sha(cp), "meta_sha256":sha(mp)})
pd.DataFrame(input_audit).to_csv(OUT/"qc/input_hashes.tsv", sep="\t", index=False)
require(0 < IV_P_THRESHOLD < 1, "P threshold must be between zero and one")

## 1. 在两个独立人员池内按同一年龄分布抽样

BP reference 用原来随机抽取的 BP 人员估计年龄分布。每侧的五种年龄构成在这个分布上做相同的指数倾斜。两侧各自按最紧的年龄箱容量确定共同样本量，所以 BMI 和 BP 的样本量可能不同。对角线上的两侧使用同一种目标年龄分布。年龄箱比例、均值和 Wasserstein 距离用于检查实际匹配。

In [ ]:
all_ids = pd.Index(sorted(set().union(*(set(y.index) for y in pheno.values()))), name="IID")
roles = pd.Series([role(i) for i in all_ids], index=all_ids, name="role")
roles.to_csv(OUT/"design/individual_roles.tsv", sep="\t", index_label="#IID")
bp_ids = pheno["SBP"].index.intersection(pheno["DBP"].index).sort_values()
require(np.allclose(age["SBP"].loc[bp_ids], age["DBP"].loc[bp_ids], atol=1e-7, rtol=0), "SBP/DBP ages differ; use separate target designs")
target_pool = bp_ids[roles.loc[bp_ids].eq("target")]
source_pool = pheno["BMI"].index[roles.loc[pheno["BMI"].index].eq("source")].sort_values()
require(min(len(target_pool), len(source_pool)) >= MIN_ROLE_N, "At least one role pool too small")
target_ids = pd.Index(stable_rng("target").choice(target_pool, min(N_TARGET_REQUESTED, len(target_pool)), replace=False)).sort_values()
require(not set(target_ids)&set(source_pool), "Role overlap")
for name, ids in [("bp_age_reference", target_ids), ("source_pool", source_pool)]:
    pd.DataFrame({"#IID":ids}).to_csv(OUT/"design"/f"{name}.keep.tsv", sep="\t", index=False)

target_age = age["SBP"].loc[target_ids].to_numpy()
source_age = age["BMI"].loc[source_pool].to_numpy()
require(np.std(target_age, ddof=1) > 0, "No target age variation")
age_min = np.floor(min(target_age.min(), source_age.min())/AGE_BIN_YEARS)*AGE_BIN_YEARS
age_max = np.floor(max(target_age.max(), source_age.max())/AGE_BIN_YEARS)*AGE_BIN_YEARS+AGE_BIN_YEARS
edges = np.arange(age_min, age_max+AGE_BIN_YEARS*.5, AGE_BIN_YEARS)
target_bin = np.searchsorted(edges, target_age, side="right")-1
source_bin = np.searchsorted(edges, source_age, side="right")-1
nbins = len(edges)-1
target_count = np.bincount(target_bin, minlength=nbins)
pool_count = np.bincount(source_bin, minlength=nbins)
pi = target_count/len(target_ids)
active = pi > 0
require((pool_count[active] > 0).all(), "Target age support missing in source pool; inspect design before changing bins")
centers = (edges[:-1]+edges[1:])/2
zcenters = (centers-target_age.mean())/target_age.std(ddof=1)
probabilities = {}
for name, tilt in SCENARIOS:
    q = pi*np.exp(tilt*zcenters)
    probabilities[name] = q/q.sum()
capacity = min(float(np.min(pool_count[active]/q[active])) for q in probabilities.values())
N_SOURCE = min(N_SOURCE_REQUESTED, int(.9*capacity))
require(N_SOURCE >= MIN_ROLE_N, "Insufficient shared age support for equal-sized sources")
if N_SOURCE < N_SOURCE_REQUESTED:
    warnings.warn(f"Common source N reduced once to {N_SOURCE} for age-bin capacity; fixed across all scenarios/repeats")

def quotas(q, n, available):
    v = n*np.asarray(q); a = np.floor(v).astype(int)
    extra = n-a.sum()
    if extra: a[np.argsort(-(v-a), kind="stable")[:extra]] += 1
    require(a.sum() == n and (a <= available).all(), "Age-bin quota exceeds available participants")
    return a

quota_rows, sample_rows, source_sets = [], [], []
for rep in range(N_REPEATS):
    # Common within-bin rankings make matched/mismatched comparisons paired.
    rankings = {b:stable_rng(f"source|{rep}|{b}").permutation(source_pool[source_bin == b]) for b in range(nbins)}
    for name, tilt in SCENARIOS:
        q = probabilities[name]; a = quotas(q, N_SOURCE, pool_count)
        ids = pd.Index(np.concatenate([rankings[b][:a[b]] for b in range(nbins)])).sort_values()
        require(ids.is_unique and len(ids) == N_SOURCE and not set(ids)&set(target_ids), "Invalid source sample")
        sid = f"r{rep+1:02d}_{name}"
        source_sets.append({"sid":sid, "repeat":rep+1, "scenario":name, "tilt":tilt, "ids":ids})
        pd.DataFrame({"#IID":ids}).to_csv(OUT/"design"/f"{sid}.keep.tsv", sep="\t", index=False)
        observed = age["BMI"].loc[ids].to_numpy()
        bindex = np.searchsorted(edges, observed, side="right")-1
        realized_q = a/N_SOURCE
        diagnostic_weights = pi[bindex]/realized_q[bindex]
        ess = diagnostic_weights.sum()**2/(diagnostic_weights@diagnostic_weights)
        uncovered_mass = float(pi[(pi>0)&(a==0)].sum())
        sample_rows.append({"sid":sid, "repeat":rep+1, "scenario":name, "tilt":tilt, "N_source":len(ids),
                            "age_mean":observed.mean(), "age_SD":observed.std(ddof=1),
                            "target_age_mean":target_age.mean(), "age_mean_gap":observed.mean()-target_age.mean(),
                            "Wasserstein_years":stats.wasserstein_distance(observed,target_age),
                            "KS_statistic":stats.ks_2samp(observed,target_age).statistic,
                            "target_probability_in_unsampled_bins":uncovered_mass,
                            "age_bin_reweight_ESS_on_observed_support_only":ess})
        for b in range(nbins):
            quota_rows.append({"sid":sid, "bin_lower":edges[b], "bin_upper":edges[b+1],
                               "source_pool_count":pool_count[b], "target_count":target_count[b],
                               "target_probability":pi[b], "intended_source_probability":q[b], "N_sampled":a[b]})
sample_qc = pd.DataFrame(sample_rows)
age_quotas = pd.DataFrame(quota_rows)
age_quotas["realized_source_probability"] = age_quotas.N_sampled/N_SOURCE
age_quotas.to_csv(OUT/"design/age_bin_quotas.tsv",sep="\t",index=False)
age_proportions = age_quotas.loc[age_quotas.sid.str.startswith("r01_")].pivot(
    index=["bin_lower","bin_upper","target_probability"],columns="sid",values="realized_source_probability")
age_proportions.to_csv(OUT/"design/age_bin_proportions_wide.tsv",sep="\t")
print("Actual age-bin proportions (same quotas across repeats):")
display(age_proportions)
sample_qc.to_csv(OUT/"design/age_mixture_QC.tsv", sep="\t", index=False)
S = len(source_sets)
overlap = np.array([[len(set(a["ids"])&set(b["ids"])) for b in source_sets] for a in source_sets])
pd.DataFrame(overlap,index=[s["sid"] for s in source_sets],columns=[s["sid"] for s in source_sets]).to_csv(OUT/"design/source_overlap_counts.tsv",sep="\t")

# Use the full BP role pool and the same age bins and tilt as BMI.
bp_pool_age = age["SBP"].loc[target_pool].to_numpy()
bp_pool_bin = np.searchsorted(edges, bp_pool_age, side="right")-1
in_bins = (bp_pool_bin >= 0) & (bp_pool_bin < nbins)
bp_pool_count = np.bincount(bp_pool_bin[in_bins], minlength=nbins)
bp_capacity = min(float(np.min(bp_pool_count[active]/q[active])) for q in probabilities.values())
N_TARGET = min(N_TARGET_REQUESTED, int(.9*bp_capacity))
require(N_TARGET >= MIN_ROLE_N, "Insufficient shared age support for equal-sized BP targets")
bp_rankings = {b: stable_rng(f"bp-target|{b}").permutation(target_pool[bp_pool_bin == b])
               for b in range(nbins)}
target_sets, bp_quota_rows = {}, []
for name, q in probabilities.items():
    counts = quotas(q, N_TARGET, bp_pool_count)
    ids = pd.Index(np.concatenate([bp_rankings[b][:counts[b]] for b in range(nbins)])).sort_values()
    require(ids.is_unique and len(ids) == N_TARGET and not set(ids)&set(source_pool),
            "Invalid BP target sample")
    target_sets[name] = ids
    pd.DataFrame({"#IID":ids}).to_csv(OUT/"design"/f"bp_{name}.keep.tsv", sep="\t", index=False)
    for b in range(nbins):
        bp_quota_rows.append({"BP_scenario":name,"bin_lower":edges[b],"bin_upper":edges[b+1],
                              "probability":counts[b]/N_TARGET,"N_sampled":counts[b],
                              "N_available":bp_pool_count[b]})
pd.DataFrame(bp_quota_rows).to_csv(OUT/"design/BP_age_bin_quotas.tsv",sep="\t",index=False)
target_union_ids = pd.Index(sorted(set().union(*(set(ids) for ids in target_sets.values()))))
bp_overlap = pd.DataFrame(
    [[len(a.intersection(b)) for b in target_sets.values()] for a in target_sets.values()],
    index=target_sets, columns=target_sets)
bp_overlap.to_csv(OUT/"design/BP_overlap_counts.tsv",sep="\t")
bp_ages = {name:age["SBP"].loc[ids].to_numpy() for name,ids in target_sets.items()}
cross_age_qc = pd.DataFrame([
    {"sid":s["sid"],"BMI_scenario":s["scenario"],"BP_scenario":name,"repeat":s["repeat"],
     "BMI_age_mean":source_ages.mean(),"BP_age_mean":bp_age.mean(),"N_BMI":N_SOURCE,"N_BP":N_TARGET,
     "age_mean_gap":source_ages.mean()-bp_age.mean(),
     "Wasserstein_years":stats.wasserstein_distance(source_ages,bp_age)}
    for s in source_sets for source_ages in [age["BMI"].loc[s["ids"]].to_numpy()]
    for name,bp_age in bp_ages.items()
])
cross_age_qc.to_csv(OUT/"design/crossed_age_QC.tsv",sep="\t",index=False)
display(cross_age_qc.groupby(["BMI_scenario","BP_scenario"]).mean(numeric_only=True)
        [["BMI_age_mean","BP_age_mean","age_mean_gap","Wasserstein_years","N_BMI","N_BP"]])

design = {"seed":SEED, "N_BP_reference":len(target_ids), "N_target":N_TARGET, "N_target_pool":len(target_pool), "N_source_each":N_SOURCE,
          "N_source_pool":len(source_pool), "repeats":N_REPEATS, "scenarios":SCENARIOS, "age_bin_years":AGE_BIN_YEARS,
          "source_P_threshold":IV_P_THRESHOLD, "reclumping":False,
          "N_boot":N_BOOT, "contrast_family":CONTRAST_FAMILY, "unrelated_keep":UNRELATED_KEEP,
          "nuisance":"saved full age spline + sex + QPC span", "target_age_mean":target_age.mean()}
write_json(OUT/"design/design.json", design)
display(pd.DataFrame([{k:design[k] for k in ("N_target","N_source_each","N_source_pool","repeats")}]))
display(sample_qc)

## 2. 每个 source 做 pooled BMI GWAS，按自己的 P 值选工具

PLINK 普通 OLS，保留完整 nuisance 空间、原始 phenotype 单位。`P < IV_P_THRESHOLD` 是唯一的工具相关性筛选；不重新 clump，不添加 F/MAF/MAC 筛选。每个 source 保存整个面板的 GWAS、所选工具清单、工具数量和工具集合重合度。无效回归明确记录；某个 source 没有工具时 MR unavailable，其他 source 继续。

In [ ]:
def full_rank_covariates(D):
    _, rankD, _ = nuisance_basis(D)
    kept=[]; X=np.ones((len(D),1)); rank=1
    for col in D:
        v=D[col].to_numpy()
        if np.std(v)<=1e-14: continue
        trial=np.column_stack([X,(v-v.mean())/v.std(ddof=1)])
        if np.linalg.matrix_rank(trial)>rank:
            kept.append(col); X=trial; rank+=1
    require(rank==rankD,"Nuisance span changed during rank reduction")
    Dc=D[kept].copy()
    return (Dc-Dc.mean())/Dc.std(ddof=1),kept,rankD

def select_source_instruments(x,threshold):
    valid=(x.ERRCODE.eq(".") & np.isfinite(x.P) & x.P.between(0,1)
           & np.isfinite(x.BETA) & np.isfinite(x.SE) & x.SE.gt(0)
           & (x.A1.eq(x.REF)|x.A1.eq(x.ALT)))
    return valid,valid & x.P.lt(threshold)

source_gwas,selection_rows={},[]
panel_lookup=panel.set_index("ID",drop=False)
for s in source_sets:
    sid,ids=s["sid"],s["ids"]
    folder=OUT/"source_GWAS"/sid; folder.mkdir(parents=True,exist_ok=True)
    pp,cp=folder/"BMI.pheno.tsv",folder/"BMI.covar.tsv"
    pheno["BMI"].loc[ids].to_csv(pp,sep="\t",index_label="#IID",header=True,float_format="%.17g")
    Dc,kept,rankD=full_rank_covariates(covar["BMI"].loc[ids])
    Dc.to_csv(cp,sep="\t",index_label="#IID",float_format="%.17g")
    write_json(folder/"input_signature.json",{"pheno_sha256":sha(pp),"covar_sha256":sha(cp),
        "keep_sha256":sha(OUT/"design"/f"{sid}.keep.tsv"),"kept_covariates":kept,
        "nuisance_rank":rankD,"P_threshold":IV_P_THRESHOLD,"reclumping":False})
    parts=[]
    for chrom in sorted(pfiles):
        chrom_panel=panel.loc[panel["#CHROM"].astype(int).eq(chrom)]
        if chrom_panel.empty: continue
        print(f"Source GWAS: {sid}, chr{chrom}",flush=True)
        prefix=folder/f"chr{chrom}"; extract=folder/f"chr{chrom}.panel.txt"
        chrom_panel.ID.to_csv(extract,index=False,header=False)
        run_cmd([PLINK,"--pfile",pfiles[chrom],"--keep",OUT/"design"/f"{sid}.keep.tsv",
            "--extract",extract,"--pheno",pp,"--pheno-name","BMI","--covar",cp,
            "--no-input-missing-phenotype","--glm","hide-covar",
            "--threads",THREADS,"--memory",MEMORY_MB,"--out",prefix],prefix)
        x=pd.read_csv(str(prefix)+".BMI.glm.linear",sep="\t",dtype=str,keep_default_na=False)
        require(x.TEST.eq("ADD").all() and x.ID.is_unique,"Expected one ADD row per SNP")
        for col in ("P","BETA","SE","OBS_CT"): x[col]=pd.to_numeric(x[col],errors="coerce")
        x["#CHROM"]=x["#CHROM"].str.replace(r"^chr","",regex=True)
        x["POS"]=pd.to_numeric(x.POS,errors="raise").astype(int)
        require(set(x.ID)==set(chrom_panel.ID),"GWAS panel coverage differs")
        for col in KEY:
            require(np.array_equal(x[col].astype(str).to_numpy(),panel_lookup.loc[x.ID,col].astype(str).to_numpy()),f"GWAS panel mismatch: {col}")
        x["F_classic"]=(x.BETA/x.SE)**2
        x["beta_ALT"]=x.BETA*np.where(x.A1.eq(x.ALT),1.,-1.)
        x["valid_GWAS"],x["selected_source_P"]=select_source_instruments(x,IV_P_THRESHOLD)
        parts.append(x)
    x=pd.concat(parts,ignore_index=True)
    require(x.ID.is_unique and set(x.ID)==set(panel.ID),"Source panel incomplete or duplicated")
    x.to_csv(folder/"pooled_BMI_GWAS.tsv",sep="\t",index=False)
    chosen=x.loc[x.selected_source_P].copy()
    chosen["BMI_increasing_allele"]=np.where(chosen.beta_ALT.gt(0),chosen.ALT,chosen.REF)
    chosen.to_csv(folder/"selected_IVs.tsv",sep="\t",index=False)
    source_gwas[sid]=x.set_index("ID",drop=False)
    s["iv_ids"]=pd.Index(chosen.ID)
    selection_rows.append({"sid":sid,"repeat":s["repeat"],"scenario":s["scenario"],
        "N_panel":len(x),"N_valid_GWAS":int(x.valid_GWAS.sum()),"N_selected":len(chosen),
        "P_threshold":IV_P_THRESHOLD,"min_F_classic":chosen.F_classic.min(),
        "mean_F_classic":chosen.F_classic.mean(),"N_selected_F_lt10":int(chosen.F_classic.lt(10).sum())})
selection=pd.DataFrame(selection_rows)
selection.to_csv(OUT/"qc/source_IV_selection.tsv",sep="\t",index=False)
display(selection)
union_ids=set().union(*(set(s["iv_ids"]) for s in source_sets))
iv=panel.loc[panel.ID.isin(union_ids),KEY].copy().sort_values(
    ["#CHROM","POS","ID"],key=lambda c:c.astype(int) if c.name in ("#CHROM","POS") else c).reset_index(drop=True)
K=len(iv)
iv.to_csv(OUT/"source_GWAS/union_selected_IVs.tsv",sep="\t",index=False)
require(K<=MAX_UNION_IV,"Union exceeds covariance memory guard; inspect counts/increase guard, never trim selected SNPs")
selection_overlap=[]
for a in source_sets:
    for b in source_sets:
        ia,ib=set(a["iv_ids"]),set(b["iv_ids"])
        selection_overlap.append({"source_a":a["sid"],"source_b":b["sid"],"K_a":len(ia),"K_b":len(ib),
            "intersection":len(ia&ib),"union":len(ia|ib),"Jaccard":len(ia&ib)/len(ia|ib) if ia|ib else np.nan})
pd.DataFrame(selection_overlap).to_csv(OUT/"qc/IV_set_overlap.tsv",sep="\t",index=False)
if K==0: warnings.warn("No significant source IV; MR unavailable. Threshold is not relaxed.")
print("Union of source-selected SNPs:",K)

## 3. 估计两侧 SNP 关联和重叠样本的协方差

各 BMI source 用自身 GWAS 的系数选工具和估计效应。用相同的 SNP 并集在各 BP 组分别回归 SBP、DBP。HC1 influence 按人对齐，用于估计同一侧样本重叠产生的协方差。

In [ ]:
# Explicit ALT-count export; default PLINK A export can count REF instead.
analysis_ids = pd.Index(sorted(set(target_union_ids).union(*(set(s["ids"]) for s in source_sets))))
pd.DataFrame({"#IID":analysis_ids}).to_csv(OUT/"design/genotype_export.keep.tsv",sep="\t",index=False)
G = pd.DataFrame(index=analysis_ids)
for chrom_s, part in iv.groupby("#CHROM",sort=False):
    chrom = int(chrom_s); prefix = OUT/"genotypes"/f"IV_chr{chrom}"
    extract = Path(str(prefix)+".extract.txt"); alleles = Path(str(prefix)+".ALT.tsv")
    part.ID.to_csv(extract,index=False,header=False)
    part[["ID","ALT"]].to_csv(alleles,sep="\t",index=False,header=False)
    run_cmd([PLINK,"--pfile",pfiles[chrom],"--keep",OUT/"design/genotype_export.keep.tsv",
             "--extract",extract,"--export","A","--export-allele",alleles,
             "--threads",THREADS,"--memory",MEMORY_MB,"--out",prefix],prefix)
    raw = pd.read_csv(str(prefix)+".raw",sep=r"\s+",dtype={"IID":str}).set_index("IID")
    require(raw.index.is_unique and set(raw.index)==set(analysis_ids),"Exported genotype IDs differ")
    if "FID" in raw: require(raw.FID.astype(str).eq("0").all(),"Unexpected genotype FID")
    for row in part.itertuples(index=False):
        col = f"{row.ID}_{row.ALT}"
        require(col in raw,f"ALT export allele unverified: {row.ID}")
        values = pd.to_numeric(raw.loc[analysis_ids,col],errors="raise")
        require(values.dropna().between(0,2).all(),"Invalid autosomal dosage")
        G[row.ID] = values
    del raw
G = G.loc[:,iv.ID]

def associations(y, C, dosage):
    """Marginal SNP regression adjusted for the full nuisance span; no Gxage.

    HC1 influence columns permit covariance across SNPs and overlapping samples.
    Each SNP uses its observed genotypes without mean imputation for regression.
    """
    y = np.asarray(y,float); C = np.asarray(C,float); A = np.asarray(dosage,float)
    n, k = A.shape
    IF = np.zeros((n,k)); rows = []; cache = {}
    for j in range(k):
        ok = np.isfinite(A[:,j]); nobs = int(ok.sum())
        token = np.packbits(ok).tobytes()
        if token not in cache:
            Q, rC, _ = nuisance_basis(C[ok])
            yr = y[ok]-Q@(Q.T@y[ok])
            cache[token] = (Q,rC,yr)
        Q, rC, yr = cache[token]
        g = A[ok,j]; gr = g-Q@(Q.T@g); den = float(gr@gr)
        df = nobs-rC-1
        row = {"N":nobs,"DF_RESID":df,"nuisance_rank":rC,"MAC":min(np.sum(g),2*nobs-np.sum(g))}
        if df<=5 or den<=1e-10 or nobs<MIN_ROLE_N:
            rows.append({**row,"status":"unidentifiable_or_too_small","BETA":np.nan,"SE_HC1":np.nan}); continue
        b = float(gr@yr/den); e = yr-b*gr
        influence = np.sqrt(nobs/df)*gr*e/den
        IF[ok,j] = influence
        se = float(np.linalg.norm(influence)); seclassic = float(np.sqrt((e@e/df)/den))
        rows.append({**row,"status":"ok" if se>0 and np.isfinite(se) else "invalid_SE", "BETA":b,"SE_HC1":se,
                     "SE_classic":seclassic,"P_HC1":2*stats.norm.sf(abs(b/se)) if se>0 else np.nan,
                     "P_classic":2*stats.t.sf(abs(b/seclassic),df) if seclassic>0 else np.nan,
                     "F_HC1":(b/se)**2 if se>0 else np.nan})
    cols=["N","DF_RESID","nuisance_rank","MAC","status","BETA","SE_HC1","SE_classic","P_HC1","P_classic","F_HC1"]
    return pd.DataFrame(rows).reindex(columns=cols), IF



target_assoc,target_cov = {},{}
for outcome in OUTCOMES:
    target_influences = []
    for name, ids in target_sets.items():
        a, influence = associations(pheno[outcome].loc[ids], covar[outcome].loc[ids], G.loc[ids])
        a = pd.concat([iv[KEY],a],axis=1)
        a["BETA_ALT"] = a.BETA
        a.to_csv(OUT/"associations"/f"bp_{name}_{outcome}_ALT.tsv",sep="\t",index=False)
        target_assoc[outcome,name] = a
        positions = target_union_ids.get_indexer(ids)
        require((positions >= 0).all(), "BP participant absent from target union")
        target_influences.append(sparse.csr_matrix(
            (influence.ravel(), (np.repeat(positions,K),np.tile(np.arange(K),len(ids)))),
            shape=(len(target_union_ids),K)))
        del influence
    target_influence = sparse.hstack(target_influences,format="csr")
    target_cov[outcome] = (target_influence.T@target_influence).toarray()
    np.save(OUT/"associations"/f"crossed_target_{outcome}_HC1_cov.npy",target_cov[outcome])
    del target_influence, target_influences
write_json(OUT/"associations/target_covariance_order.json",
           {"BP_scenarios":list(target_sets),"IV_order":iv.ID.tolist()})

source_assoc,sparse_IF,strength_rows,audit_rows=[],[],[],[]
for s in source_sets:
    sid,ids,chosen=s["sid"],s["ids"],s["iv_ids"]
    a,influence=associations(pheno["BMI"].loc[ids],covar["BMI"].loc[ids],G.loc[ids,chosen])
    a=pd.concat([panel_lookup.loc[chosen,KEY].reset_index(drop=True),a],axis=1)
    require(a.status.eq("ok").all(),f"{sid}: selected source SNP refit failed")
    expected=source_gwas[sid].loc[chosen]
    beta_ok=np.allclose(a.BETA,expected.beta_ALT,rtol=5e-4,atol=5e-6)
    se_ok=np.allclose(a.SE_classic,expected.SE,rtol=5e-4,atol=5e-6)
    n_ok=np.array_equal(a.N.to_numpy(),expected.OBS_CT.to_numpy())
    audit_rows.append({"sid":sid,"K":len(a),"beta_agrees":beta_ok,"SE_agrees":se_ok,"N_agrees":n_ok})
    pd.DataFrame(audit_rows).to_csv(OUT/"qc/source_GWAS_refit_agreement.tsv",sep="\t",index=False)
    require(beta_ok and se_ok and n_ok,f"{sid}: GWAS/refit disagreement, inspect before MR")
    a["BETA_ALT"]=a.BETA
    a["orientation"]=np.where(expected.beta_ALT.to_numpy()>0,1.,-1.)
    a["BETA"]=expected.beta_ALT.to_numpy()*a.orientation
    a["SE_classic"]=expected.SE.to_numpy()
    a["P_GWAS"]=expected.P.to_numpy()
    a["sid"]=sid
    influence*=a.orientation.to_numpy()[None,:]
    a.to_csv(OUT/"associations"/f"{sid}_selected_BMI.tsv",sep="\t",index=False)
    source_assoc.append(a)
    ii=source_pool.get_indexer(ids); k=len(a)
    require((ii>=0).all(),"Source ID absent from pool")
    sparse_IF.append(sparse.csr_matrix((influence.ravel(),(np.repeat(ii,k),np.tile(np.arange(k),len(ii)))),shape=(len(source_pool),k)))
    strength_rows.append({"sid":sid,"K_selected":k,"mean_F_HC1":a.F_HC1.mean(),"min_F_HC1":a.F_HC1.min(),
        "N_F_HC1_lt10":int(a.F_HC1.lt(10).sum()),"min_MAC":a.MAC.min()})
    del influence
source_strength=pd.DataFrame(strength_rows)
source_strength.to_csv(OUT/"qc/source_IV_strength.tsv",sep="\t",index=False)
offsets=np.cumsum([0]+[len(a) for a in source_assoc])
IF_all=sparse.hstack(sparse_IF,format="csr")
SigmaX=(IF_all.T@IF_all).toarray()
require(np.allclose(np.sqrt(np.diag(SigmaX)),np.concatenate([a.SE_HC1.to_numpy() for a in source_assoc]),rtol=1e-9,atol=1e-12),"Source covariance diagonal mismatch")
np.save(OUT/"associations/source_joint_HC1_cov.npy",SigmaX)
write_json(OUT/"associations/source_covariance_order.json",{"sources":[
    {"sid":s["sid"],"IV_order":a.ID.tolist(),"start":int(offsets[j]),"end":int(offsets[j+1])}
    for j,(s,a) in enumerate(zip(source_sets,source_assoc))]})
del sparse_IF,IF_all,G
gc.collect()
display(source_strength)

## 4. 计算 25 个组合和配对差异

每个组合报告 IVW、dIVW、weighted median 和 MR-Egger。Median 和 Egger 在本实现中需要至少 3 个可用 SNP。重复平均需要三个重复都有估计。dIVW 使用 BMI GWAS 的普通 SE 校正 IVW 分母。同一 BMI GWAS 用于筛选和估计效应，winner's curse 仍可能影响两种估计。

配对重抽使用 BMI 和 BP 两侧的联合协方差，各侧对所有组合重用同一套系数抽样。对固定 BP 组比较不同 BMI 组与 BMI matched，对固定 BMI 组比较不同 BP 组与 BP matched。区间条件于观测到的工具集合，重抽时固定这些集合。两个结局、四种方法和两个方向共计划 320 个平均差异，使用 Bonferroni 校正。

In [ ]:
def weighted_median(x,y,se_y):
    x,y,se_y = np.asarray(x,float),np.asarray(y,float),np.asarray(se_y,float)
    w = x*x/(se_y*se_y)
    ok = np.isfinite(w)&(w>0)&np.isfinite(x)&np.isfinite(y)
    if not ok.any(): return np.nan
    ratio = y[ok]/x[ok]; w = w[ok]
    order = np.argsort(ratio,kind="stable"); ratio = ratio[order]; w = w[order]
    positions = (np.cumsum(w)-.5*w)/w.sum()
    return float(np.interp(.5,positions,ratio))

def mr_points(x,y,se_y,se_x):
    """Batch estimates; orientation fixed to each source observed BMI-increasing alleles."""
    X = np.atleast_2d(x); Y = np.atleast_2d(y)
    X,Y = np.broadcast_arrays(X,Y)
    if X.shape[1]==0: return {m:np.full(X.shape[0],np.nan) for m in METHODS}
    w = 1/np.asarray(se_y,float)**2
    numerator = np.sum(w*X*Y,axis=1)
    den = np.sum(w*X*X,axis=1)
    ivw = np.divide(numerator,den,out=np.full(len(X),np.nan),where=den>1e-20)
    # Ye, Shao & Kang (2021), equation (12), applied to the observed selected set.
    corrected_denominator = den - np.sum(w*np.asarray(se_x,float)**2)
    divw = np.divide(numerator,corrected_denominator,out=np.full(len(X),np.nan),
                     where=np.isfinite(corrected_denominator)&(corrected_denominator>1e-20))
    median = np.full(len(X),np.nan); egger = np.full(len(X),np.nan)
    if X.shape[1]>=3:
        median = np.array([weighted_median(a,b,se_y) for a,b in zip(X,Y)])
        sw = w.sum(); xm = np.sum(w*X,axis=1)/sw; ym = np.sum(w*Y,axis=1)/sw
        d = np.sum(w*(X-xm[:,None])**2,axis=1)
        egger = np.divide(np.sum(w*(X-xm[:,None])*(Y-ym[:,None]),axis=1),d,
                          out=np.full(len(X),np.nan),where=d>1e-20)
    return {"IVW":ivw,"dIVW":divw,"weighted_median":median,"MR_Egger":egger}

def mr_diagnostics(x,sx,y,sy):
    w = 1/sy**2; k = len(x); d = np.sum(w*x*x)
    theta = np.sum(w*x*y)/d
    Q = np.sum(w*(y-theta*x)**2)
    fese = 1/np.sqrt(d)
    rese = fese*np.sqrt(max(1,Q/(k-1))) if k>1 else fese
    out = {"IVW_Q":Q,"IVW_Q_df":k-1,"IVW_Q_P":stats.chi2.sf(Q,k-1) if k>1 else np.nan,
           "IVW_SE_FE_NOME":fese,"IVW_SE_MRE_NOME":rese,"max_IVW_weight_fraction":np.max(w*x*x)/d,
           "IVW_classic_CI_low":theta-1.959964*rese,"IVW_classic_CI_high":theta+1.959964*rese}
    if k>1:
        wx = 1/sx**2; bx = np.sum(wx*x)/wx.sum(); qx = np.sum(wx*(x-bx)**2)
        out["I2_GX"] = max(0,1-(k-1)/qx) if qx>0 else 0.
    else: out["I2_GX"] = np.nan
    if k>=3:
        sw = w.sum(); xm = np.sum(w*x)/sw; ym = np.sum(w*y)/sw
        dd = np.sum(w*(x-xm)**2)
        if dd>1e-20:
            slope = np.sum(w*(x-xm)*(y-ym))/dd; intercept = ym-slope*xm
            qe = np.sum(w*(y-intercept-slope*x)**2); scale = max(1,qe/(k-2))
            sei = np.sqrt(scale*(1/sw+xm*xm/dd)); ses = np.sqrt(scale/dd)
            out.update({"Egger_intercept":intercept,"Egger_intercept_SE_MRE_NOME":sei,
                        "Egger_intercept_P_normal":2*stats.norm.sf(abs(intercept/sei)),"Egger_SE_MRE_NOME":ses,
                        "Egger_Q":qe,"Egger_Q_df":k-2,"Egger_Q_P":stats.chi2.sf(qe,k-2)})
    return out

def covariance_root(V):
    if V.size==0: return V.copy()
    V = (V+V.T)/2
    lam,U = linalg.eigh(V,check_finite=False)
    require(lam.min()>=-1e-7*max(lam.max(),1e-20),"Covariance has materially negative eigenvalues")
    return U*np.sqrt(np.maximum(lam,0))[None,:]

def uncertainty(point,draws):
    good = draws[np.isfinite(draws)]
    if not np.isfinite(point) or len(good)<.99*len(draws):
        return {"status":"unavailable_or_unstable","SE_joint":np.nan,"CI_low":np.nan,"CI_high":np.nan,"P_joint_normal":np.nan,"N_boot_valid":len(good)}
    se = np.std(good,ddof=1)
    return {"status":"ok","SE_joint":se,"CI_low":point-1.959964*se,"CI_high":point+1.959964*se,
            "P_joint_normal":2*stats.norm.sf(abs(point/se)) if se>0 else np.nan,
            "bootstrap_percentile_low":np.quantile(good,.025),"bootstrap_percentile_high":np.quantile(good,.975),
            "bootstrap_median_minus_point":np.median(good)-point,"N_boot_valid":len(good)}


def conventional_mr(x,sx,y,sy):
    k=len(x)
    points={m:float(v[0]) for m,v in mr_points(x,y,sy,sx).items()}
    diag=mr_diagnostics(x,sx,y,sy) if k else {}
    divw_se = np.nan
    if k:
        w_hat = (x/sy)**2
        v_hat = (sx/sy)**2
        corrected_denominator = np.sum(w_hat-v_hat)
        diag["dIVW_denominator"] = corrected_denominator
        if np.isfinite(points["dIVW"]):
            # Reference variance, equation (14). Same-source selection is not covered.
            variance_numerator = np.sum(w_hat + points["dIVW"]**2*v_hat*(w_hat+v_hat))
            divw_se = np.sqrt(variance_numerator)/corrected_denominator

    standard_errors = {"IVW":diag.get("IVW_SE_MRE_NOME",np.nan),
                       "dIVW":divw_se,
                       "MR_Egger":diag.get("Egger_SE_MRE_NOME",np.nan)}
    inference = {"IVW":"first-order MRE/NOME", "MR_Egger":"first-order MRE/NOME",
                 "dIVW":"equation (14) reference SE with same-source screening",
                 "weighted_median":"see conditional sensitivity columns"}
    rows={}
    for method,point in points.items():
        se = standard_errors.get(method,np.nan)
        status = "ok" if np.isfinite(point) else "no_or_too_few_IVs"
        if method=="dIVW" and k and not np.isfinite(point):
            status = "unstable_dIVW_denominator"
        rows[method]={"estimate":point,"status":status,
            "SE_conventional":se,"CI_low_conventional":point-1.959964*se,"CI_high_conventional":point+1.959964*se,
            "P_conventional_normal":2*stats.norm.sf(abs(point/se)) if se>0 else np.nan,
            "inference":inference[method]}
    return rows,diag


# Joint draws preserve overlapping groups on each side, conditional on selected SNPs.
Xhat = np.concatenate([a.BETA.to_numpy() for a in source_assoc])
rng = stable_rng("crossed-joint-coefficients")
drawX = Xhat[None,:] + rng.standard_normal((N_BOOT,len(Xhat))) @ covariance_root(SigmaX).T
mr_rows, mean_rows, contrast_rows, harmonization_rows = [], [], [], []
target_names = list(target_sets)
scenario_names = [name for name,_ in SCENARIOS]
source_indices = {(s["scenario"],s["repeat"]):j for j,s in enumerate(source_sets)}
for outcome in OUTCOMES:
    valid_target = {}
    target_means = []
    for name in target_names:
        target = target_assoc[outcome,name]
        valid = target.status.eq("ok") & np.isfinite(target.BETA) & target.SE_classic.gt(0)
        valid_target[name] = valid.to_numpy()
        target_means.append(np.where(valid,target.BETA,0.))
    yhat = np.concatenate(target_means)
    drawY = yhat[None,:] + rng.standard_normal((N_BOOT,len(yhat))) @ covariance_root(target_cov[outcome]).T
    estimates = {method:np.full((len(target_names),S),np.nan) for method in METHODS}
    draws = {method:np.full((len(target_names),S,N_BOOT),np.nan) for method in METHODS}
    used_counts = np.zeros((len(target_names),S),dtype=int)
    for t, name in enumerate(target_names):
        target = target_assoc[outcome,name]
        for j,(s,a) in enumerate(zip(source_sets,source_assoc)):
            idx = pd.Index(iv.ID).get_indexer(a.ID)
            require((idx>=0).all(),"Source IV absent from union")
            ok = valid_target[name][idx]
            ia, iy = np.flatnonzero(ok), idx[ok]
            orient = a.orientation.to_numpy()[ok]
            x, sx = a.BETA.to_numpy()[ok], a.SE_classic.to_numpy()[ok]
            y, sy = target.BETA.to_numpy()[iy]*orient, target.SE_classic.to_numpy()[iy]
            resampled = mr_points(drawX[:,offsets[j]:offsets[j+1]][:,ia],
                                  drawY[:,t*K+iy]*orient,sy,sx)
            conventional, diag = conventional_mr(x,sx,y,sy)
            used_counts[t,j] = len(x)
            group = {"outcome":outcome,"sid":s["sid"],"BMI_scenario":s["scenario"],
                     "BP_scenario":name,"repeat":s["repeat"]}
            harmonization_rows.append({**group,"K_selected":len(a),"K_used":len(x),
                                       "N_invalid_target":int((~ok).sum())})
            for method in METHODS:
                point = conventional[method]["estimate"]
                estimates[method][t,j] = point
                draws[method][t,j] = resampled[method]
                sensitivity = {"conditional_"+key:value
                               for key,value in uncertainty(point,resampled[method]).items()}
                mr_rows.append({**group,"method":method,"K_selected":len(a),"K_used":len(x),
                                **conventional[method],**sensitivity,**diag})
    np.savez_compressed(OUT/"MR"/f"{outcome}_conditional_crossed_draws.npz",**draws)
    write_json(OUT/"MR"/f"{outcome}_draw_order.json",
               {"BP_scenarios":target_names,"BMI_sources":[s["sid"] for s in source_sets]})
    matched_target = target_names.index("matched")
    matched_sources = [source_indices["matched",r] for r in range(1,N_REPEATS+1)]
    for t,name in enumerate(target_names):
        for scenario in scenario_names:
            jj = [source_indices[scenario,r] for r in range(1,N_REPEATS+1)]
            for method in METHODS:
                points = estimates[method][t,jj]
                boot = draws[method][t,jj]
                group = {"outcome":outcome,"BMI_scenario":scenario,"BP_scenario":name,"method":method}
                mean = float(np.mean(points))
                mean_rows.append({**group,"estimate_mean_repeats":mean,
                                  "K_used_min":int(used_counts[t,jj].min()),
                                  "K_used_max":int(used_counts[t,jj].max()),
                                  "N_repeats_available":int(np.isfinite(points).sum()),
                                  **uncertainty(mean,boot.mean(axis=0))})
                baselines = []
                if scenario != "matched":
                    baselines.append(("BMI",t,matched_sources))
                if name != "matched":
                    baselines.append(("BP",matched_target,jj))
                for axis, ref_t, ref_j in baselines:
                    differences = points - estimates[method][ref_t,ref_j]
                    boot_differences = boot - draws[method][ref_t,ref_j]
                    delta = float(differences.mean())
                    contrast_rows.append({**group,"varying_side":axis,"delta":delta,
                                          **uncertainty(delta,boot_differences.mean(axis=0))})
mr = pd.DataFrame(mr_rows)
mean_mr = pd.DataFrame(mean_rows)
contrasts = pd.DataFrame(contrast_rows)
contrasts["P_Bonf_conditional"] = np.minimum(contrasts.P_joint_normal*CONTRAST_FAMILY,1)
pd.DataFrame(harmonization_rows).to_csv(OUT/"qc/target_harmonization.tsv",sep="\t",index=False)
mr.to_csv(OUT/"MR/MR_crossed_all_repeats.tsv",sep="\t",index=False)
mean_mr.to_csv(OUT/"MR/MR_crossed_mean_repeats_conditional.tsv",sep="\t",index=False)
contrasts.to_csv(OUT/"MR/MR_crossed_paired_contrasts_conditional.tsv",sep="\t",index=False)
display(mean_mr)

## 5. 读取交叉结果

图中行是 BMI 年龄方案，列是 BP 年龄方案。每格显示三次 BMI 重复的平均 MR 点估计和条件 SE。每个组合的实际年龄差可查 design/crossed_age_QC.tsv。按列比较可以固定 BP，按行比较可以固定 BMI。对应的配对区间在 MR/MR_crossed_paired_contrasts_conditional.tsv。

In [ ]:
order = [name for name,_ in sorted(SCENARIOS,key=lambda item:item[1])]
labels = ["Younger strong","Younger moderate","Matched","Older moderate","Older strong"]
for outcome in OUTCOMES:
    rows = mean_mr.loc[mean_mr.outcome.eq(outcome)]
    points = rows.estimate_mean_repeats.to_numpy()
    limit = max(np.max(np.abs(points[np.isfinite(points)]),initial=0.),1e-6)
    fig,axes = plt.subplots(2,2,figsize=(13,10),layout="constrained")
    for ax,method in zip(axes.flat,METHODS):
        grid = rows.loc[rows.method.eq(method)]
        point = grid.pivot(index="BMI_scenario",columns="BP_scenario",values="estimate_mean_repeats").reindex(index=order,columns=order)
        se = grid.pivot(index="BMI_scenario",columns="BP_scenario",values="SE_joint").reindex(index=order,columns=order)
        image = ax.imshow(point.to_numpy(),cmap="RdBu_r",vmin=-limit,vmax=limit)
        for i,j in np.ndindex(point.shape):
            estimate, stderr = point.iloc[i,j], se.iloc[i,j]
            label = f"{estimate:.3f}" if np.isfinite(estimate) else "NA"
            if np.isfinite(stderr):
                label += f"\n({stderr:.3f})"
            color = "white" if np.isfinite(estimate) and abs(estimate)>.65*limit else "black"
            ax.text(j,i,label,ha="center",va="center",color=color,fontsize=8)
        ax.set_xticks(range(5),labels,rotation=35,ha="right")
        ax.set_yticks(range(5),labels)
        ax.set(title=method,xlabel="BP age distribution",ylabel="BMI age distribution")
    fig.colorbar(image,ax=axes,label="mmHg per BMI unit",shrink=.8)
    fig.suptitle(f"BMI → {outcome} | Estimate (conditional SE)")
    fig.savefig(OUT/"figures"/f"MR_crossed_{outcome}.png",dpi=180)
    plt.show()
write_json(OUT/"ANALYSIS_COMPLETE.json",{
    "status":"complete","finished_utc":datetime.now(timezone.utc).isoformat(),
    "design":design,"linear_run":str(LINEAR_RUN),"plink_version":PLINK_VERSION,
    "K_union":K,"N_MR_rows":len(mr),"N_mean_contrasts":len(contrasts),
    "selection_uncertainty_included":False})
print("B3 crossed analysis completed:",OUT)